In [16]:
import pandas as pd
import numpy as np
import os as os
import re

In [17]:
csvs = ['US1ILAD0005.csv', 'USC00110072.csv', 'USC00110338.csv', 'USC00111329.csv',
        'USC00111577.csv', 'USC00114489.csv', 'USC00114629.csv', 'USC00115097.csv',
        'USC00115493.csv', 'USC00116200.csv', 'USC00117077.csv', 'USC00117391.csv',
        'USC00118293.csv', 'USC00118740.csv', 'USW00003887.csv', 'USW00014842.csv', 
        'USW00093822.csv', 'USW00094822.csv', 'USW00094870.csv']

PATH = '/Users/Carl/Documents/gitfiles/final-project-clara-and-roshni/data/raw-data/Weather data'

all_dfs = []

for file in csvs:
    full_path = os.path.join(PATH, file)
    df = pd.read_csv(full_path)
    
    # Optionally: add a column for station id or cleaned name
    raw_name = df.loc[0, 'NAME']
    clean_name = raw_name.lower()
    clean_name = clean_name.replace(" ", "_")
    clean_name = re.sub(r'[^a-z0-9_]', '', clean_name)
    df['station_name'] = clean_name
    df['station_id'] = file.replace('.csv','')
    
    all_dfs.append(df)

# Combine all into one master dataframe
master_df = pd.concat(all_dfs, ignore_index=True)

# Optional: drop rows without TMAX or TMIN
master_df = master_df.dropna(subset=['TMAX', 'TMIN'])

print(master_df.head())
print(f"Total rows: {len(master_df)}")


/var/folders/rz/fxhdpdkx56b9hbb6wt0vdlsw0000gn/T/ipykernel_32883/943621608.py:13: DtypeWarning: Columns (19,23,37,39,41,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(full_path)
/var/folders/rz/fxhdpdkx56b9hbb6wt0vdlsw0000gn/T/ipykernel_32883/943621608.py:13: DtypeWarning: Columns (19,21,25,29,41,43,45,49,51,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(full_path)
/var/folders/rz/fxhdpdkx56b9hbb6wt0vdlsw0000gn/T/ipykernel_32883/943621608.py:13: DtypeWarning: Columns (17,19,21,23,35,39,41,43) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(full_path)
/var/folders/rz/fxhdpdkx56b9hbb6wt0vdlsw0000gn/T/ipykernel_32883/943621608.py:13: DtypeWarning: Columns (17,19,21,23,25,27,29,31,35,37,39,41,43,45,49,59,61,63,65,67,69,71) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(full_path)
/var/folders/rz/fxhdpd

         STATION        DATE  LATITUDE  LONGITUDE  ELEVATION          NAME  \
641  USC00110072  1901-01-01  41.20693  -90.74187      223.7  ALEDO, IL US   
642  USC00110072  1901-01-02  41.20693  -90.74187      223.7  ALEDO, IL US   
643  USC00110072  1901-01-03  41.20693  -90.74187      223.7  ALEDO, IL US   
644  USC00110072  1901-01-04  41.20693  -90.74187      223.7  ALEDO, IL US   
645  USC00110072  1901-01-05  41.20693  -90.74187      223.7  ALEDO, IL US   

     PRCP PRCP_ATTRIBUTES  SNOW SNOW_ATTRIBUTES  ...  WV20 WV20_ATTRIBUTES  \
641   0.0            ,,0,   0.0            ,,0,  ...   NaN             NaN   
642   0.0            ,,0,   0.0            ,,0,  ...   NaN             NaN   
643   0.0            ,,0,   0.0            ,,0,  ...   NaN             NaN   
644   0.0            ,,0,   0.0            ,,0,  ...   NaN             NaN   
645   0.0            ,,0,   0.0            ,,0,  ...   NaN             NaN   

     SN32 SN32_ATTRIBUTES  SX32 SX32_ATTRIBUTES  WT12 WT12_ATT

In [18]:
# Get inventory file to find COOP stations in IL
inv_path = os.path.join(PATH, 'ghcnd-inventory.txt')
inv = pd.read_fwf(inv_path, 
                    colspecs=[(0,11),(12,20),(21,30),(31,35),(36,40),(41,45)],
                    names=["station_id","lat","lon","element","first_year","last_year"])

# Filter for COOP stations (IDs starting with 'USC')
coop_stations = inv[inv['station_id'].str.startswith('USC')]

# Filter for Illinois by lat/lon
coop_il = coop_stations[
    (coop_stations['lat'] >= 36.98) &
    (coop_stations['lat'] <= 42.50) &
    (coop_stations['lon'] >= -91.52) &
    (coop_stations['lon'] <= -87.51)
]

# Keep only rows with TMAX or TMIN
coop_il_temp = coop_il[coop_il['element'].isin(['TMAX','TMIN'])]

# Only stations that have BOTH TMAX and TMIN
stations_with_both = (
    coop_il_temp.groupby('station_id')['element']
    .nunique()
    .loc[lambda x: x == 2]
    .index
)

# Now filter for stations with data covering 2018–2025
coop_il_recent = coop_il_temp[
    coop_il_temp['station_id'].isin(stations_with_both) &
    (coop_il_temp['first_year'] <= 2018) &
    (coop_il_temp['last_year'] >= 2025)
]

# Get unique station IDs
stations_recent_ids = coop_il_recent['station_id'].unique()

print("Illinois COOP stations with TMAX & TMIN covering 2018–2025:")
print(stations_recent_ids)

Illinois COOP stations with TMAX & TMIN covering 2018–2025:
['USC00110072' 'USC00110137' 'USC00110187' 'USC00110338' 'USC00110356'
 'USC00110442' 'USC00110598' 'USC00111265' 'USC00111290' 'USC00111329'
 'USC00111436' 'USC00111550' 'USC00111577' 'USC00111836' 'USC00112048'
 'USC00112140' 'USC00112193' 'USC00112223' 'USC00112348' 'USC00112353'
 'USC00112483' 'USC00112500' 'USC00112736' 'USC00112745' 'USC00112931'
 'USC00113106' 'USC00113262' 'USC00113290' 'USC00113320' 'USC00113335'
 'USC00113384' 'USC00113455' 'USC00113580' 'USC00113879' 'USC00114012'
 'USC00114108' 'USC00114198' 'USC00114355' 'USC00114400' 'USC00114442'
 'USC00114447' 'USC00114489' 'USC00114530' 'USC00114603' 'USC00114629'
 'USC00114710' 'USC00114780' 'USC00114823' 'USC00114957' 'USC00115079'
 'USC00115097' 'USC00115372' 'USC00115493' 'USC00115712' 'USC00115768'
 'USC00115772' 'USC00115825' 'USC00115833' 'USC00115841' 'USC00115901'
 'USC00115943' 'USC00116080' 'USC00116157' 'USC00116200' 'USC00116344'
 'USC00116446' 'U

In [19]:
import geopandas as gpd
from shapely.geometry import Point

target_counties = [
    'Adams', 'Champaign', 'Clark', 'Cook', 'DuPage',
    'Effingham', 'Hamilton', 'Jersey', 'Jo Daviess', 'Kane',
    'Lake', 'Macon', 'Macoupin', 'Madison', 'McHenry',
    'McLean', 'Peoria', 'Randolph', 'Rock Island', 'Saint Clair',
    'Sangamon', 'Will', 'Winnebago'
]

counties_path = os.path.join(PATH, 'tl_2025_us_county')
counties = gpd.read_file(counties_path)
counties = counties[counties['NAME'].isin(target_counties)]

# Load inventory and filter for COOP + coverage
inv_path = os.path.join(PATH, 'ghcnd-inventory.txt')
inv = pd.read_fwf(
    inv_path,
    colspecs=[(0,11),(12,20),(21,30),(31,35),(36,40),(41,45)],
    names=["station_id","lat","lon","element","first_year","last_year"]
)

coop = inv[inv.station_id.str.startswith("USC")]
coop_il = coop[(coop.lat>=36.98)&(coop.lat<=42.50)&(coop.lon>=-91.52)&(coop.lon<=-87.51)]
coop_temp = coop_il[coop_il.element.isin(['TMAX','TMIN'])]
stations_both = coop_temp.groupby('station_id')['element'].nunique()
stations_both = stations_both[stations_both == 2].index

valid = coop_temp[
    (coop_temp.station_id.isin(stations_both)) &
    (coop_temp.first_year <= 2018) &
    (coop_temp.last_year >= 2025)
].drop_duplicates('station_id')

stations_gdf = gpd.GeoDataFrame(
    valid[['station_id','lat','lon','first_year','last_year']],
    geometry=gpd.points_from_xy(valid.lon, valid.lat),
    crs="EPSG:4326"
)

In [20]:
selected = []

for _, county in counties.iterrows():
    # stations within this county
    inside = stations_gdf[stations_gdf.within(county.geometry)]
    
    if len(inside) == 0:
        # If none, you could find nearest nearby later
        continue
    
    # pick one inside county
    # e.g., most complete coverage → longest span
    inside['span'] = inside['last_year'] - inside['first_year']
    best = inside.sort_values('span', ascending=False).iloc[0]
    
    selected.append({
        'county': county['NAME'],
        'station_id': best['station_id'],
        'lat': best['lat'],
        'lon': best['lon'],
        'first_year': best['first_year'],
        'last_year': best['last_year']
    })

final_df = pd.DataFrame(selected)
print(final_df)

/opt/miniconda3/envs/dap/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/opt/miniconda3/envs/dap/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/opt/miniconda3/envs/dap/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

         county   station_id      lat      lon  first_year  last_year
0        DuPage  USC00115097  41.8128 -88.0728        2007       2026
1      Randolph  USC00114629  37.9836 -89.9469        1974       2026
2         Adams  USC00117077  39.9036 -91.4283        1937       2026
3          Will  USC00114530  41.5033 -88.1033        1975       2026
4        McLean  USC00116200  40.5492 -88.9497        1977       2026
5     Champaign  USC00118740  40.0842 -88.2403        1902       2026
6          Cook  USC00111577  41.7372 -87.7772        1928       2026
7         Clark  USC00111329  39.2975 -87.9747        1893       2026
8          Kane  USC00110338  41.7803 -88.3092        1893       2026
9    Jo Daviess  USC00118293  42.3997 -89.9903        1943       2026
10      Madison  USC00110137  38.8669 -90.1489        1943       2026
11        Macon  USC00112193  39.8289 -88.9506        1893       2026
12       Jersey  USC00114489  39.1025 -90.3431        1940       2026
13  Rock Island  USC

/opt/miniconda3/envs/dap/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/opt/miniconda3/envs/dap/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/opt/miniconda3/envs/dap/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

In [21]:
# Define function to convert columns to fahrenheit
def c_to_f_inplace(df, col_name):
    """
    Convert a dataframe column from Celsius to Fahrenheit in place.

    Parameters
    ----------
    df : pandas.DataFrame
        The dataframe containing the column
    col_name : str
        Name of the column to convert

    Returns
    -------
    pandas.DataFrame
        The same dataframe with the column converted
    """
    
    df[col_name] = (df[col_name]/10) * 9/5 + 32
    return df

master_df = c_to_f_inplace(master_df, 'TMIN')
master_df = c_to_f_inplace(master_df, 'TMAX')
master_df['tavg_calc'] = (master_df['TMIN'] + master_df['TMAX'])/2

/var/folders/rz/fxhdpdkx56b9hbb6wt0vdlsw0000gn/T/ipykernel_32883/3417332664.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  master_df['tavg_calc'] = (master_df['TMIN'] + master_df['TMAX'])/2


In [22]:
master_df

,STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,PRCP,PRCP_ATTRIBUTES,SNOW,SNOW_ATTRIBUTES,...,WV20_ATTRIBUTES,SN32,SN32_ATTRIBUTES,SX32,SX32_ATTRIBUTES,WT12,WT12_ATTRIBUTES,WV18,WV18_ATTRIBUTES,tavg_calc
641,USC00110072,1901-01-01,41.20693,-90.74187,223.7,"ALEDO, IL US",0.0,",,0,",0.0,",,0,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.51
642,USC00110072,1901-01-02,41.20693,-90.74187,223.7,"ALEDO, IL US",0.0,",,0,",0.0,",,0,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.99
643,USC00110072,1901-01-03,41.20693,-90.74187,223.7,"ALEDO, IL US",0.0,",,0,",0.0,",,0,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.03
644,USC00110072,1901-01-04,41.20693,-90.74187,223.7,"ALEDO, IL US",0.0,",,0,",0.0,",,0,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.98
645,USC00110072,1901-01-05,41.20693,-90.74187,223.7,"ALEDO, IL US",0.0,",,0,",0.0,",,0,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
504348,USW00094870,2026-02-18,40.03240,-88.27547,226.5,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",0.0,",,D",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.51
504349,USW00094870,2026-02-19,40.03240,-88.27547,226.5,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",8.0,",,D",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.99
504350,USW00094870,2026-02-20,40.03240,-88.27547,226.5,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",0.0,",,D",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.96
504351,USW00094870,2026-02-21,40.03240,-88.27547,226.5,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",0.0,",,D",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.96


In [23]:
col_keep = ['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'NAME', 'TMAX', 'TMIN', 'tavg_calc']

master_df = master_df[col_keep]

master_df

,STATION,DATE,LATITUDE,LONGITUDE,NAME,TMAX,TMIN,tavg_calc
641,USC00110072,1901-01-01,41.20693,-90.74187,"ALEDO, IL US",21.02,-4.00,8.51
642,USC00110072,1901-01-02,41.20693,-90.74187,"ALEDO, IL US",26.96,3.02,14.99
643,USC00110072,1901-01-03,41.20693,-90.74187,"ALEDO, IL US",33.98,6.08,20.03
644,USC00110072,1901-01-04,41.20693,-90.74187,"ALEDO, IL US",41.00,26.96,33.98
645,USC00110072,1901-01-05,41.20693,-90.74187,"ALEDO, IL US",33.08,23.00,28.04
...,...,...,...,...,...,...,...,...
504348,USW00094870,2026-02-18,40.03240,-88.27547,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",68.00,39.02,53.51
504349,USW00094870,2026-02-19,40.03240,-88.27547,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",60.98,41.00,50.99
504350,USW00094870,2026-02-20,40.03240,-88.27547,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",50.00,21.92,35.96
504351,USW00094870,2026-02-21,40.03240,-88.27547,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",35.96,17.96,26.96


In [24]:
master_df = master_df[master_df['DATE'] >= '2015-01-01']

master_df

,STATION,DATE,LATITUDE,LONGITUDE,NAME,TMAX,TMIN,tavg_calc
42155,USC00110072,2015-01-01,41.20693,-90.74187,"ALEDO, IL US",21.02,-0.04,10.49
42156,USC00110072,2015-01-02,41.20693,-90.74187,"ALEDO, IL US",33.08,14.00,23.54
42157,USC00110072,2015-01-03,41.20693,-90.74187,"ALEDO, IL US",35.96,15.98,25.97
42158,USC00110072,2015-01-04,41.20693,-90.74187,"ALEDO, IL US",33.08,19.94,26.51
42159,USC00110072,2015-01-05,41.20693,-90.74187,"ALEDO, IL US",19.94,-2.02,8.96
...,...,...,...,...,...,...,...,...
504348,USW00094870,2026-02-18,40.03240,-88.27547,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",68.00,39.02,53.51
504349,USW00094870,2026-02-19,40.03240,-88.27547,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",60.98,41.00,50.99
504350,USW00094870,2026-02-20,40.03240,-88.27547,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",50.00,21.92,35.96
504351,USW00094870,2026-02-21,40.03240,-88.27547,"CHAMPAIGN URBANA WILLARD AIRPORT, IL US",35.96,17.96,26.96


In [25]:
master_df.to_csv("all_weather.csv", index=False)